# Wearable SLR Analysis Notebook


In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from matplotlib.patches import Ellipse
from matplotlib.ticker import ScalarFormatter

# --- GLOBAL JOURNAL STYLING ---
plt.close('all')
sns.set_theme(style="whitegrid")

GLOBAL_FONT_SIZE = 22
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.size": GLOBAL_FONT_SIZE,
    "axes.titlesize": GLOBAL_FONT_SIZE + 2,
    "axes.labelsize": GLOBAL_FONT_SIZE,
    "xtick.labelsize": GLOBAL_FONT_SIZE - 2,
    "ytick.labelsize": GLOBAL_FONT_SIZE - 2,
    "legend.fontsize": GLOBAL_FONT_SIZE - 2,
    "figure.dpi": 400,
    "axes.titleweight": "bold"
})

GLOBAL_PALETTE = "viridis"
PAL_VIRIDIS = sns.color_palette("viridis", 5)
PAL_MAGMA = sns.color_palette("magma", 5)
FIG_SIZE = (16, 6) # Uniform height and width for all figures

def add_subplot_labels(axes, y_offset=-0.25):
    """Adds (a), (b), (c) labels to the center-bottom of subplots."""
    if not isinstance(axes, (list, np.ndarray)):
        axes = [axes]
    else:
        axes = np.array(axes).flatten()
    
    if len(axes) > 1:
        labels = ['(a)', '(b)', '(c)', '(d)']
        for idx, ax in enumerate(axes):
            # Use the y_offset variable here
            ax.text(0.5, y_offset, labels[idx], transform=ax.transAxes, 
                    ha='center', va='top', fontsize=GLOBAL_FONT_SIZE+2, fontweight='bold')


In [ ]:
# --- 1. LOAD DATA & CLEANING ---
json_folder = 'jsons_test'
data_list = []
if os.path.exists(json_folder):
    for filename in os.listdir(json_folder):
        if filename.endswith('.json'):
            with open(os.path.join(json_folder, filename), 'r') as f:
                content = json.load(f)
                if content.get('relevance') == 'include':
                    data_list.append(content)
print(f"Successfully loaded {len(data_list)} relevant papers.")

def to_float_safe(val):
    if val is None: return None
    if isinstance(val, list): val = val[0] if len(val) > 0 else None
    if isinstance(val, dict): val = val.get('value', val.get('eer', val.get('result')))
    try: return float(val)
    except (ValueError, TypeError): return None

def compute_repro_score(repr_info):
    score = 0
    if repr_info.get('code_available'): score += 2
    if repr_info.get('datasets_all_public'): score += 2
    elif repr_info.get('datasets_any_public'): score += 1
    if repr_info.get('benchmark_used'): score += 1
    return score

# Minimal required mapping based on the original structure
# Note: This mapping can be expanded based on the actual variety of labels found in the dataset. The goal is to standardize similar concepts under a unified label for better analysis.
normalization_map = {
    'ECG': 'ECG', 'PPG': 'PPG', 'Accelerometer': 'Accelerometer', 'Gyroscope': 'Gyroscope',
    'IMU': 'IMU', 'motion': 'Motion', 'EDA': 'EDA/GSR', 'EMG': 'EMG', 'EEG': 'EEG',
    'Voice': 'Voice/Acoustic', 'speech': 'Voice/Acoustic', 'acoustic': 'Voice/Acoustic',
    'Keystroke dynamics': 'Keystroke', 'typing': 'Keystroke', 'touch': 'Touch/Swipe',
    'Gesture': 'Gestures', 'gait': 'Gait', 'fingerprint': 'Physical Biometrics',
    'CNN': 'CNN', 'ResNet': 'ResNet', 'LSTM': 'LSTM', 'RNN': 'RNN', 'SVM': 'SVM',
    'Random Forest': 'Random Forest', 'Decision Tree': 'Decision Tree',
    'K-Nearest Neighbor': 'k-NN', 'DNN': 'DNN/MLP'
}




def clean_labels(label_list):
    if not isinstance(label_list, list): return []
    return list(set([normalization_map.get(item.strip(), item.strip()) for item in label_list if item]))

# Base Flattening
rows = []
for p in data_list:
    eval_metrics = {m['metric'].upper(): m['value'] for m in p.get('evaluation', []) if m['value'] is not None}
    gap = p.get('real_world_gap') or {}
    temp = p.get('temporal_robustness') or {}
    perf = p.get('performance_analysis') or {}
    sec = p.get('security') or {}
    use = p.get('usability') or {}
    repr_info = p.get('reproducibility') or {}
    
    rows.append({
        'title': p.get('title'),
        'year': pd.to_numeric(p.get('year'), errors='coerce'),
        'biometric_type': str(p.get('biometric', {}).get('type')).strip(),
        'modalities': clean_labels(p.get('biometric', {}).get('modality', [])),
        'learning_type': str(p.get('method', {}).get('learning_type')).strip(),
        'models': clean_labels(p.get('method', {}).get('model', [])),
        'eer': to_float_safe(eval_metrics.get('EER')),
        'rw_tested': bool(p.get('real_world_tested', False)),
        'lab_eer': to_float_safe(gap.get('lab_performance_value')),
        'wild_eer': to_float_safe(gap.get('real_world_performance_value')),
        'Cross-Session': bool(temp.get('cross_session_tested')),
        'Motion Robustness': bool((perf.get('robust_to_motion') or {}).get('is_quantified')),
        'Noise Robustness': bool((perf.get('robust_to_noise') or {}).get('is_quantified')),
        'gap_tags': p.get('insights', {}).get('gap_category') or [],
        'spoofing': bool(sec.get('spoofing_tested')),
        'liveness': bool(sec.get('liveness_detection')),
        'attacks': sec.get('attack_types') or [],
        'repro_raw': compute_repro_score(repr_info)
    })
df = pd.DataFrame(rows)
df = df.dropna(subset=['year']).copy()
df['year'] = df['year'].astype(int)


In [ ]:
# --- RQ1 FIGURES ---

# FIGURE 1: Biometric Type & Learning Type vs EER
fig, axes = plt.subplots(1, 2, figsize=FIG_SIZE)
sns.countplot(data=df, x='biometric_type', ax=axes[0], palette=GLOBAL_PALETTE, hue='biometric_type', legend=False)
axes[0].set_xlabel("Biometric Category")
axes[0].set_ylabel("Number of Papers")

eer_df = df[df['eer'].notnull()].copy()
sns.boxplot(data=eer_df, x='learning_type', y='eer', ax=axes[1], palette=GLOBAL_PALETTE, hue='learning_type', legend=False)
axes[1].set_xlabel("Learning Type")
axes[1].set_ylabel("Equal Error Rate (EER)")
axes[1].set_ylim(0, eer_df['eer'].quantile(0.95))
axes[1].tick_params(axis='x', rotation=20)

add_subplot_labels(axes, y_offset=-0.47)
plt.tight_layout()
plt.savefig('RQ1_Fig1_Categories.png', dpi=400, bbox_inches='tight')
plt.close()

# FIGURE 2: Top Modalities & Models
fig, axes = plt.subplots(1, 2, figsize=FIG_SIZE)
mod_counts = df.explode('modalities')['modalities'].value_counts().head(10)
sns.barplot(x=mod_counts.values, y=mod_counts.index, ax=axes[0], palette=GLOBAL_PALETTE, hue=mod_counts.index, legend=False)
axes[0].set_xlabel("Number of Papers")
axes[0].set_ylabel("Modality")

model_counts = df.explode('models')['models'].value_counts().head(10)
sns.barplot(x=model_counts.values, y=model_counts.index, ax=axes[1], palette=GLOBAL_PALETTE, hue=model_counts.index, legend=False)
axes[1].set_xlabel("Number of Papers")
axes[1].set_ylabel("Model Architecture")

add_subplot_labels(axes)
plt.tight_layout()
plt.savefig('RQ1_Fig2_TopTech.png', dpi=400, bbox_inches='tight')
plt.close()

# FIGURE 3: Reproducibility Score (RQ1 Score Plot)
df_score = df[df['year'].between(2017, 2026)].copy()
fig, axes = plt.subplots(1, 3, figsize=FIG_SIZE)
sns.histplot(df_score['repro_raw'], bins=6, kde=False, ax=axes[0], color=PAL_MAGMA[2])
axes[0].set_xlabel("Reproducibility Score")
axes[0].set_ylabel("Number of Papers")

sns.boxplot(data=df_score, x='year', y='repro_raw', palette=GLOBAL_PALETTE, hue='year', legend=False, ax=axes[1])
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Score")
axes[1].tick_params(axis='x', rotation=45)

mean_scores = df_score.groupby('year')['repro_raw'].mean().reset_index()
sns.lineplot(data=mean_scores, color='black', x='year', y='repro_raw', marker='o', ax=axes[2], linewidth=3)
axes[2].set_xlabel("Year")
axes[2].set_ylabel("Mean Score")
axes[2].set_ylim(0, 5)
axes[2].set_xticks(mean_scores['year'].astype(int))
axes[2].tick_params(axis='x', rotation=45)

add_subplot_labels(axes, y_offset=-0.47)
plt.tight_layout()
plt.savefig('RQ1_Fig_3_Score.png', dpi=400, bbox_inches='tight')
plt.close()


In [ ]:
# --- RQ2 FIGURES ---
# FIGURE 1: Testing Landscape
fig, axes = plt.subplots(1, 3, figsize=FIG_SIZE)

# 1. Pie Chart
rw_counts = df['rw_tested'].map({True: 'Real-World', False: 'Lab-Only'}).value_counts()
axes[0].pie(rw_counts, labels=rw_counts.index, autopct='%1.1f%%', colors=[PAL_VIRIDIS[1], PAL_VIRIDIS[4]], startangle=140)

# --- NEW: Fix the "2017.0" issue by forcing integers ---
df_clean_year = df.dropna(subset=['year']).copy()
df_clean_year['year'] = df_clean_year['year'].astype(int)
# -------------------------------------------------------

# 2. Bar Chart
env_trend = pd.crosstab(df_clean_year['year'], df_clean_year['rw_tested']).rename(columns={True: 'Real-World', False: 'Lab-Only'})
env_trend.plot(kind='bar', stacked=True, ax=axes[1], color=[PAL_VIRIDIS[0], PAL_VIRIDIS[3]], alpha=0.8)
axes[1].set_ylabel("Number of Papers")
axes[1].set_xlabel("Year")
axes[1].tick_params(axis='x', rotation=45)

# --- NEW: Push the legend cleanly below the X-axis ---
axes[1].legend(
    title="", 
    loc="upper center",          # Anchors the top of the legend...
    bbox_to_anchor=(0.5, -0.35), # ...to a point far below the x-axis
    ncol=2, 
    frameon=False
)
# -----------------------------------------------------

# 3. Line Chart
line_data = (env_trend['Real-World'] / env_trend.sum(axis=1) * 100)
sns.lineplot(x=line_data.index.astype(str), y=line_data.values, marker='o', ax=axes[2], color='teal', linewidth=3)
axes[2].set_ylabel("Real-World Testing (%)")
axes[2].set_xlabel("Year")
axes[2].set_ylim(0, 105)
axes[2].tick_params(axis='x', rotation=45)

# --- NEW: Use your y_offset to push (a)(b)(c) below everything ---
add_subplot_labels(axes, y_offset=-0.55) 
# -----------------------------------------------------------------

plt.tight_layout()
plt.savefig('RQ2_Fig1_Testing_Landscape.png', dpi=400, bbox_inches='tight')
plt.close()

# FIGURE 2: Performance Penalty (EER Gap)
fig, axes = plt.subplots(1, 2, figsize=FIG_SIZE)
def categorize_eer(row):
    if pd.notnull(row['lab_eer']) and pd.notnull(row['wild_eer']): return 'Both EERs'
    if pd.notnull(row['lab_eer']) or pd.notnull(row['wild_eer']): return 'Single EER'
    return 'No EER'

avail = df.apply(categorize_eer, axis=1).value_counts()
sns.barplot(x=avail.index, y=avail.values, ax=axes[0], palette=GLOBAL_PALETTE, hue=avail.index, legend=False)
axes[0].set_ylabel("Paper Count")
axes[0].set_xlabel("Data Availability")
#axes[0].tick_params(axis='x', rotation=45)

scatter_df = df.dropna(subset=['lab_eer', 'wild_eer']).copy()
scatter_df = scatter_df[(scatter_df['lab_eer'] <= 20) & (scatter_df['wild_eer'] <= 20)]
sns.scatterplot(data=scatter_df, x='lab_eer', y='wild_eer', s=150, palette=GLOBAL_PALETTE, ax=axes[1], alpha=0.6)
axes[1].plot([0, 20], [0, 20], 'k--', alpha=0.4)
axes[1].fill_between([0, 20], [0, 20], 20, color='red', alpha=0.05)
axes[1].set_xlim(0, 20); axes[1].set_ylim(0, 20)
axes[1].set_xlabel("Lab EER (%)"); axes[1].set_ylabel("Real-World EER (%)")

add_subplot_labels(axes)
plt.tight_layout()
plt.savefig('RQ2_Fig2_Performance_Penalty.png', dpi=400, bbox_inches='tight')
plt.close()

# FIGURE 3: Robustness Evolution
fig, axes = plt.subplots(1, 2, figsize=FIG_SIZE)
rob_cols = ['Motion Robustness', 'Noise Robustness', 'Cross-Session']
volume_df = df.groupby('year')[rob_cols].sum()
volume_df.plot(kind='bar', stacked=True, ax=axes[0], color=PAL_VIRIDIS[:3], alpha=0.85)
axes[0].set_ylabel("Number of Papers")
axes[0].set_xlabel("Year")
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(
    title="",
    fontsize=18,
    handlelength=0.8,   # shorter rectangles
    handleheight=0.4,   # shorter vertically
    labelspacing=0.2,   # less vertical gap
    borderpad=0.2,
    handletextpad=0.3
)

yearly_trends = df.groupby('year')[rob_cols].mean() * 100
markers = ['o', 's', '^']
for i, column in enumerate(rob_cols):
    sns.lineplot(x=yearly_trends.index.astype(str), y=yearly_trends[column], 
                 marker=markers[i], markersize=10, color=PAL_VIRIDIS[i], linewidth=3, label=column, ax=axes[1])
axes[1].set_ylabel("% of Annual Papers (%)")
axes[1].set_xlabel("Year")
axes[1].set_ylim(0, 105)
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(
    title="",
    fontsize=19,
    handlelength=0.8,   # shorter rectangles
    handleheight=0.6,   # shorter vertically
    labelspacing=0.2,   # less vertical gap
    borderpad=0.2,
    handletextpad=0.3
)

add_subplot_labels(axes, y_offset=-0.45)
plt.tight_layout()
plt.savefig('RQ2_Fig3_Robustness_Evolution.png', dpi=400, bbox_inches='tight')
plt.close()

# FIGURE 4: Subjects vs EER (Subpool)
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
corr_rows = []
for p in data_list:
    total_subs = sum([to_float_safe(ds.get('num_subjects')) for ds in p.get('data', []) if to_float_safe(ds.get('num_subjects'))])
    eer_val = next((to_float_safe(e.get('value')) for e in p.get('evaluation', []) if str(e.get('metric')).upper() == 'EER'), None)
    if eer_val and total_subs > 0:
        if eer_val < 1.0: eer_val *= 100
        if 0 < eer_val <= 20:
            corr_rows.append({'num_subjects': total_subs, 'eer': eer_val})
df_corr = pd.DataFrame(corr_rows)

if not df_corr.empty:
    sns.regplot(data=df_corr, x='num_subjects', y='eer', ax=ax, scatter_kws={'alpha': 0.7, 's': 100}, line_kws={'lw': 3})
    ax.set_xscale('log')
    ax.xaxis.set_major_formatter(ScalarFormatter())
    x_log = np.log10(df_corr['num_subjects'])
    center_x, y_mean = 10 ** x_log.mean(), df_corr['eer'].mean()
    width, height = 10 ** (x_log.mean() + x_log.std()) - 10 ** (x_log.mean() - x_log.std()), 2 * df_corr['eer'].std()
    ellipse = Ellipse((center_x, y_mean), width=width, height=height, edgecolor='black', facecolor='none', linestyle='--', linewidth=2)
    ax.add_patch(ellipse)
    ax.set_xlabel("Total Number of Subjects (Log Scale)")
    ax.set_ylabel("Standardized EER (%)")

plt.tight_layout()
plt.savefig('RQ2_Fig4_subpool.png', dpi=400, bbox_inches='tight')
plt.close()


In [ ]:
# --- RQ3 FIGURES ---

# FIGURE 1: Security Evolution
df_valid_yr = df.dropna(subset=['year']).copy()
fig, axes = plt.subplots(1, 3, figsize=FIG_SIZE)
sec_summary = df_valid_yr[['spoofing', 'liveness']].sum()
sns.barplot(x=['Spoofing', 'Liveness'], y=sec_summary.values, ax=axes[0], palette=PAL_VIRIDIS[:2], hue=['Spoofing', 'Liveness'], legend=False)
axes[0].set_ylabel("Number of Papers")

sec_vol = df_valid_yr.groupby('year')[['spoofing', 'liveness']].sum()
sec_vol.plot(kind='bar', stacked=True, ax=axes[1], color=[PAL_VIRIDIS[0], PAL_VIRIDIS[2]], alpha=0.85)
axes[1].set_ylabel("Number of Papers")
axes[1].set_xlabel("Year")
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(
    title="",
    fontsize=18,
    handlelength=0.8,   # shorter rectangles
    handleheight=0.6,   # shorter vertically
    labelspacing=0.2,   # less vertical gap
    borderpad=0.2,
    handletextpad=0.3
)

sec_pct = df_valid_yr.groupby('year')[['spoofing', 'liveness']].mean() * 100
sns.lineplot(data=sec_pct, markers=True, dashes=False, palette=[PAL_VIRIDIS[0], PAL_VIRIDIS[2]], linewidth=3, ax=axes[2])
axes[2].set_ylabel("Annual Prevalence (%)")
axes[2].set_xlabel("Year")
axes[2].set_ylim(0, 105)
axes[2].set_xticks(sec_pct.index.astype(int))
axes[2].tick_params(axis='x', rotation=45)
axes[2].legend(
    title="",
    fontsize=18,
    handlelength=0.8,   # shorter rectangles
    handleheight=0.6,   # shorter vertically
    labelspacing=0.2,   # less vertical gap
    borderpad=0.2,
    handletextpad=0.3
)

add_subplot_labels(axes, y_offset=-0.50)
plt.tight_layout()
plt.savefig('RQ3_Fig1_Security_Evolution.png', dpi=400, bbox_inches='tight')
plt.close()

# FIGURE 2: Reproducibility Improved
def score_to_label(s):
    if s >= 4: return 'high'
    elif s >= 2: return 'medium'
    return 'low'
df_valid_yr['repro_cat'] = df_valid_yr['repro_raw'].apply(score_to_label)

fig, axes = plt.subplots(1, 3, figsize=FIG_SIZE)
repro_prop = df_valid_yr['repro_cat'].value_counts(normalize=True).reindex(['low', 'medium', 'high'])
sns.barplot(x=repro_prop.index, y=repro_prop.values * 100, palette=[PAL_VIRIDIS[0], PAL_VIRIDIS[1], PAL_VIRIDIS[3]], hue=repro_prop.index, legend=False, ax=axes[0])
axes[0].set_ylabel("Percentage of Papers (%)")
axes[0].set_ylim(0, 100)

repro_trend = pd.crosstab(df_valid_yr['year'], df_valid_yr['repro_cat']).reindex(columns=['low', 'medium', 'high'], fill_value=0)
repro_trend.plot(kind='bar', stacked=True, ax=axes[1], color=[PAL_VIRIDIS[0], PAL_VIRIDIS[1], PAL_VIRIDIS[3]], alpha=0.9)
axes[1].set_ylabel("Number of Papers")
axes[1].set_xlabel("Year")
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(
    title="",
    fontsize=17,
    handlelength=0.8,   # shorter rectangles
    handleheight=0.6,   # shorter vertically
    labelspacing=0.2,   # less vertical gap
    borderpad=0.2,
    handletextpad=0.3
)

repro_stats = df_valid_yr.groupby('year')['repro_raw'].agg(['mean', 'std']).reset_index()
axes[2].errorbar(x=repro_stats['year'].astype(str), y=repro_stats['mean'], yerr=repro_stats['std'], 
                 fmt='-o', color='black', linewidth=2, capsize=4)
axes[2].set_ylabel("Mean Reproducibility \n Score")
axes[2].set_xlabel("Year")
axes[2].set_ylim(0, 5)
axes[2].tick_params(axis='x', rotation=45)

add_subplot_labels(axes, y_offset=-0.50)
plt.tight_layout()
plt.savefig('RQ3_Fig2_Reproducibility_Improved.png', dpi=400, bbox_inches='tight')
plt.close()

# FIGURE 3: Deployment Attacks
fig, axes = plt.subplots(1, 2, figsize=FIG_SIZE)
all_attacks = [a for sublist in df_valid_yr['attacks'] for a in (sublist if isinstance(sublist, list) else [sublist]) if a != 'none']
if all_attacks:
    attack_counts = pd.Series(all_attacks).value_counts().head(6)
    sns.barplot(x=attack_counts.values, y=attack_counts.index, ax=axes[0], palette=GLOBAL_PALETTE, hue=attack_counts.index, legend=False)
    axes[0].set_xlabel("Number of Mentions")
    
    attack_list = []
    for _, row in df_valid_yr.iterrows():
        for a in (row['attacks'] if isinstance(row['attacks'], list) else [row['attacks']]):
            if a and a != 'none': attack_list.append({'year': row['year'], 'attack': a})
    df_atk = pd.DataFrame(attack_list)
    top_atk_names = df_atk['attack'].value_counts().head(5).index
    df_atk_filtered = df_atk[df_atk['attack'].isin(top_atk_names)]
    
    atk_matrix = pd.crosstab(df_atk_filtered['attack'], df_atk_filtered['year']).div(df_valid_yr['year'].value_counts(), axis=1) * 100
    sns.heatmap(atk_matrix.fillna(0), annot=False, cmap="viridis", ax=axes[1], cbar_kws={'label': '% of Papers'})
    axes[1].set_ylabel("")
    axes[1].tick_params(axis='x', rotation=45)
    
add_subplot_labels(axes, y_offset=-0.60) 
plt.tight_layout()
plt.savefig('RQ3_Fig3_Deployment_Attacks.png', dpi=400, bbox_inches='tight')
plt.close()

print("All 10 figures have been successfully generated!")